# Timing FPGA-inference

In [8]:
bitfile_path = 'Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile/system.bit'
#bitfile_path = 'Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_axistream/system.bit'

x_test_path = 'Data/x_test.npy'
y_test_path = 'Data/y_test.npy'

timings_path = 'Timings/FPGA/'
iterations = 100

Load datasets for testing performance in inference

In [9]:
import numpy as np
# Load input from .npy file
x_test = np.load(x_test_path).astype(np.float32)
y_test = np.load(y_test_path).astype(np.float32)
x_test.dtype

dtype('float32')

In [10]:
def cal_accuracy(y_dut):
    y_pred = np.argmax(y_dut, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return np.sum(y_pred == y_true) / len(y_true)

In [11]:
def run_inference():
    result = overlay.predict(x_test, debug=False, profile=True, encode=np.float32, decode=np.float32)
    # calculate acc to check things work out initially
    acc = cal_accuracy(result[0])
    print(f"\naccuracy of hardware inference: {acc}")
    return result[1]

Run the actual inference

In [12]:
import os
import time
from axi_master_driver import NeuralNetworkOverlay
#from axi_stream_driver import NeuralNetworkOverlay

timestamp = time.strftime("%Y%m%d_%H%M%S")
nr_samples = x_test.shape[0]
timings = []

# create the overlay object
overlay = NeuralNetworkOverlay(bitfile_name=bitfile_path, x_shape=x_test.shape, y_shape=y_test.shape, dtype=x_test.dtype)

for i in range(iterations):
    # Do the prediction/run inference
    timings.append(run_inference())
    


timing_results_path = f"{timings_path}timings_AXISTREAM_dataset-{nr_samples}_{iterations}iterations_{timestamp}.txt"
np.savetxt(timing_results_path,timings)



accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of

In [13]:
# Compute statistics for collected timings (convert to ms)
import json
import numpy as np

arr = np.array(timings)

if arr.size == 0:
    raise RuntimeError('Timings array is empty')

arr_ms = arr * 1e3

stats = {
    'count': int(arr_ms.size),
    'mean_ms': float(np.mean(arr_ms)),
    'median_ms': float(np.median(arr_ms)),
    'std_ms': float(np.std(arr_ms, ddof=0)),
    'min_ms': float(np.min(arr_ms)),
    'max_ms': float(np.max(arr_ms)),
    'p5_ms': float(np.percentile(arr_ms, 5)),
    'p95_ms': float(np.percentile(arr_ms, 95)),
    'inference-rate (MHz)': float((nr_samples * 1000 / np.median(arr_ms) / 1000000)),
}

# Print summary
print('Timing statistics (ms):')
for k,v in stats.items():
    print(f"{k}: {v}")

# Save JSON summary next to timings file if available
try:
    base = timing_results_path
    out = base.rsplit('.',1)[0] + '_stats.json'
except NameError:
    out = 'timing_stats.json'

with open(out, 'w') as f:
    json.dump(stats, f, indent=2)

print(f'Saved timing summary to: {out}')


Timing statistics (ms):
count: 100
mean_ms: 0.47620336003092234
median_ms: 0.4568349986584508
std_ms: 0.12336367773591272
min_ms: 0.4221540002617985
max_ms: 1.604216000487213
p5_ms: 0.4296589999285061
p95_ms: 0.549861001309182
inference-rate (MHz): 1816.848539269959
Saved timing summary to: Timings/FPGA/timings_AXISTREAM_dataset-830000_100iterations_20260609_143724_stats.json


In [7]:
from pynq import PL
PL.reset()